# Long-Term Survival Analysis: 90-Day Equipment Stop Prediction

**Complementary to 4-24h short-term model**

## Purpose
While the short-term model (Phase 1-4) predicts imminent failures using real-time sensor data, this pipeline predicts the probability of equipment stops over the next **90 days** (extended from 14 days) for strategic maintenance planning.

## Key Differences from Short-Term Model

| Aspect | Short-Term (4-24h) | Long-Term (90-day) |
|--------|-------------------|-------------------|
| **Granularity** | 15-min bins | Daily aggregates |
| **Features** | Instantaneous + hourly rolling | Daily mean/max/std, weekly trends, degradation slopes |
| **Signal** | Sudden anomalies (vibration spike) | Slow drift (bearing wear, fouling, efficiency decline) |
| **Data source** | Eventhouse real-time | Lakehouse historical |
| **Additional inputs** | PI + iCare only | + operating hours, load patterns, maintenance history |
| **Output** | Binary: stop in next 4h/8h/24h | **Regression**: days-until-stop (0-90) |
| **Scoring** | Every 15 min | Daily (morning briefing) |
| **Use case** | Emergency dispatch | Strategic planning, parts procurement |

## Pipeline Phases
1. **Phase 1b** - Daily label construction with `days_to_next_stop`
2. **Phase 2b** - Daily feature engineering (trends, degradation rates, cumulative hours)
3. **Phase 3b** - Train regression model (predict days-until-stop)
4. **Phase 4b** - Daily scoring and output to `ml.predictions_shortterm`

## Phase 1b: Daily Label Construction (90-Day Horizon)

Create a daily label grid with `days_to_next_stop` (continuous 0-90) instead of binary stop/no-stop.

**Approach:**
- For each day in the historical window, calculate how many days until the next GADS stop event
- If no stop within 90 days → label = 90 (censored)
- If stop in 30 days → label = 30
- This creates a regression target for survival modeling

**Why 90 days?** Provides realistic survival curves that drop below 50% (median), unlike 14-day model where all predictions hit the censoring ceiling.

In [ ]:
# Phase 1b: Daily Label Construction with days_to_next_stop
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime, timedelta

# Parameters
TARGET_ASSETS = ['RV2_U2_Boiler', 'RV3_U3_Steam_Turbine', 'RV3_U3_Boiler_Feed_Pump_East']
HORIZON_DAYS = 90  # Predict stops within next 90 days (extended from 14)
HISTORICAL_WINDOW_DAYS = 365  # Use last year of data

print(f"Long-term survival analysis parameters (EXTENDED HORIZON):")
print(f"  Assets: {TARGET_ASSETS}")
print(f"  Prediction horizon: {HORIZON_DAYS} days (90-day outlook)")
print(f"  Historical window: {HISTORICAL_WINDOW_DAYS} days")

# Load GADS stop events (same as Phase 1 short-term)
gads_events = spark.table("gold.fact_gads_event")

# Filter to target assets and FORCED outages only (align with Phase 1 short-term pipeline)
# Exclude: PO (planned), RS (reserve shutdown), D1-D4/D (deratings)
stop_events = gads_events.filter(
    F.col("asset_id").isin(TARGET_ASSETS) &
    F.col("EVENT_TYPE_CD").isin(['U1', 'U2', 'U3', 'MO', 'SF', 'D1', 'D3'])
).select(
    F.col("asset_id"),
    F.col("REAL_START_DT").cast("date").alias("stop_date"),
    F.col("EVENT_TYPE_CD").alias("event_type"),
    F.col("CAUSE_OF_EVENT").alias("event_description")
).distinct()

print(f"\n✓ Loaded GADS stop events: {stop_events.count():,} events")
stop_events.groupBy("asset_id").count().show()

# Generate daily date grid for each asset (last HISTORICAL_WINDOW_DAYS days)
end_date = datetime.now().date()
start_date = end_date - timedelta(days=HISTORICAL_WINDOW_DAYS)

date_range = spark.range(
    0, HISTORICAL_WINDOW_DAYS + 1
).select(
    F.expr(f"date_add(date('{start_date}'), cast(id as int))").alias("calendar_date")
)

# Cross-join with assets to get daily grid
asset_dates = spark.createDataFrame(
    [(asset,) for asset in TARGET_ASSETS],
    ["asset_id"]
).crossJoin(date_range)

print(f"\n✓ Generated daily grid: {asset_dates.count():,} asset-days")
print(f"  Date range: {start_date} to {end_date}")

# For each asset-day, calculate days_to_next_stop
# Window function: look forward to find next stop
w_future = Window.partitionBy("asset_id").orderBy("calendar_date").rowsBetween(
    Window.currentRow, Window.unboundedFollowing
)

# Join asset-days with stop events
labels_with_stops = asset_dates.join(
    stop_events.select("asset_id", "stop_date"),
    (asset_dates.asset_id == stop_events.asset_id) &
    (stop_events.stop_date >= asset_dates.calendar_date),
    "left"
).select(
    asset_dates.asset_id,
    asset_dates.calendar_date,
    stop_events.stop_date
)

# Calculate days to each future stop, then take minimum (next stop)
labels_with_days = labels_with_stops.withColumn(
    "days_to_stop",
    F.datediff(F.col("stop_date"), F.col("calendar_date"))
).groupBy("asset_id", "calendar_date").agg(
    F.min("days_to_stop").alias("days_to_next_stop")
)

# Cap at HORIZON_DAYS (censored observations)
# NULL means no stop found → censor at 14 days
labels_daily = labels_with_days.withColumn(
    "days_to_next_stop",
    F.when(F.col("days_to_next_stop").isNull(), F.lit(HORIZON_DAYS))
     .when(F.col("days_to_next_stop") > HORIZON_DAYS, F.lit(HORIZON_DAYS))
     .otherwise(F.col("days_to_next_stop"))
).withColumn(
    "censored",
    F.when(F.col("days_to_next_stop") >= HORIZON_DAYS, F.lit(1)).otherwise(F.lit(0))
)

# Add binary labels for intermediate horizons (useful for monitoring)
labels_daily = labels_daily.withColumn("label_stop_7d", 
    F.when(F.col("days_to_next_stop") <= 7, 1).otherwise(0)
).withColumn("label_stop_14d",
    F.when(F.col("days_to_next_stop") <= 14, 1).otherwise(0)
)

# Save to gold schema
labels_daily.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold.daily_survival_labels")

print(f"\n✓ Saved gold.daily_survival_labels: {labels_daily.count():,} rows")
print(f"\nLabel distribution:")
labels_daily.groupBy("asset_id").agg(
    F.count("*").alias("total_days"),
    F.avg("days_to_next_stop").alias("avg_days_to_stop"),
    F.sum(F.when(F.col("label_stop_7d") == 1, 1).otherwise(0)).alias("stops_within_7d"),
    F.sum(F.when(F.col("label_stop_14d") == 1, 1).otherwise(0)).alias("stops_within_14d"),
    F.sum("censored").alias("censored_days")
).show(truncate=False)

# Preview sample
print("\nSample labels (recent days):")
labels_daily.filter(F.col("calendar_date") >= F.date_sub(F.current_date(), 30)) \
    .orderBy("asset_id", F.desc("calendar_date")).show(20, truncate=False)

## Phase 2b: Daily Feature Engineering

Create daily aggregates and trend features for long-term prediction.

**Feature types:**
1. **Daily statistics** - min/max/mean/std for each sensor (per day)
2. **7-day trends** - rolling averages and slopes
3. **14-day trends** - rolling averages and slopes
4. **Degradation indicators** - efficiency decline, variability increase
5. **Cumulative metrics** - operating hours, cumulative load
6. **Temporal features** - day_of_week, days_since_last_stop

In [ ]:
# Phase 2b: Daily Feature Engineering
from pyspark.sql import Window
from pyspark.sql import functions as F

# Load PI fact table and tag mapping
pi_fact = spark.table("gold.fact_pi")
tag_mapping = spark.table("gold.bridge_pi_tag_to_asset")

# Filter to target assets
asset_tags = tag_mapping.filter(F.col("asset_id").isin(TARGET_ASSETS))

# Join PI data with asset mapping
pi_asset = pi_fact.join(
    asset_tags.select(F.col("Tag"), F.col("asset_id").alias("bridge_asset_id")),
    "Tag",
    "inner"
).select(
    F.coalesce(F.col("asset_id"), F.col("bridge_asset_id")).alias("asset_id"),
    "Tag",
    "Timestamp",
    "ValueNumeric"
)

print(f"✓ Loaded PI data: {pi_asset.count():,} rows")
print(f"✓ Tags per asset:")
asset_tags.groupBy("asset_id").count().show(truncate=False)

# Convert timestamp to date for daily aggregation
pi_daily_base = pi_asset.withColumn(
    "calendar_date",
    F.to_date(F.col("Timestamp"))
).select(
    "asset_id",
    "Tag", 
    "calendar_date",
    "ValueNumeric"
)

# 1. Daily Statistics per tag (min/max/mean/std)
print("\n1. Computing daily statistics per tag...")

daily_stats = pi_daily_base.groupBy("asset_id", "calendar_date", "Tag").agg(
    F.min("ValueNumeric").alias("daily_min"),
    F.max("ValueNumeric").alias("daily_max"),
    F.avg("ValueNumeric").alias("daily_mean"),
    F.stddev("ValueNumeric").alias("daily_std"),
    F.count("ValueNumeric").alias("daily_count")
)

print(f"   ✓ Daily stats: {daily_stats.count():,} asset-day-tag rows")

# Pivot to wide format (each tag becomes 4 columns: min/max/mean/std)
# First, get distinct tags
distinct_tags = asset_tags.select("Tag").distinct().collect()
tag_list_raw = [row.Tag for row in distinct_tags]

# Sanitize tag names to avoid dots/colons in pivoted column names
import re
tag_list = [re.sub(r'[^a-zA-Z0-9_]', '_', t) for t in tag_list_raw]
daily_stats = daily_stats.withColumn("safe_tag", F.regexp_replace(F.col("Tag"), "[^a-zA-Z0-9_]", "_"))

print(f"   ✓ Pivoting {len(tag_list)} tags (sanitized names)...")

# Create separate dataframes for each statistic type (pivot on safe_tag)
daily_min_pivot = daily_stats.groupBy("asset_id", "calendar_date").pivot("safe_tag", tag_list).agg(F.first("daily_min"))
daily_max_pivot = daily_stats.groupBy("asset_id", "calendar_date").pivot("safe_tag", tag_list).agg(F.first("daily_max"))
daily_mean_pivot = daily_stats.groupBy("asset_id", "calendar_date").pivot("safe_tag", tag_list).agg(F.first("daily_mean"))
daily_std_pivot = daily_stats.groupBy("asset_id", "calendar_date").pivot("safe_tag", tag_list).agg(F.first("daily_std"))

# Rename columns with suffixes (tag names already sanitized)
for tag in tag_list:
    daily_min_pivot = daily_min_pivot.withColumnRenamed(tag, f"{tag}_daily_min")
    daily_max_pivot = daily_max_pivot.withColumnRenamed(tag, f"{tag}_daily_max")
    daily_mean_pivot = daily_mean_pivot.withColumnRenamed(tag, f"{tag}_daily_mean")
    daily_std_pivot = daily_std_pivot.withColumnRenamed(tag, f"{tag}_daily_std")

# Join all pivots together
daily_features = daily_min_pivot \
    .join(daily_max_pivot, ["asset_id", "calendar_date"], "inner") \
    .join(daily_mean_pivot, ["asset_id", "calendar_date"], "inner") \
    .join(daily_std_pivot, ["asset_id", "calendar_date"], "inner")

print(f"   ✓ Daily features: {daily_features.count():,} rows × {len(daily_features.columns)} columns")

# 2. Add rolling 7-day and 14-day windows
print("\n2. Computing rolling trends (7d and 14d)...")

w_7d = Window.partitionBy("asset_id").orderBy("calendar_date").rowsBetween(-6, 0)
w_14d = Window.partitionBy("asset_id").orderBy("calendar_date").rowsBetween(-13, 0)

# For key tags, compute rolling averages and slopes
# (Use a subset of tags to keep feature count manageable)
KEY_TAGS = tag_list[:10]  # Top 10 most important tags per asset

for tag in KEY_TAGS:
    mean_col = f"{tag}_daily_mean"
    
    if mean_col in daily_features.columns:
        # 7-day rolling average
        daily_features = daily_features.withColumn(
            f"{tag}_7d_avg",
            F.avg(mean_col).over(w_7d)
        )
        
        # 14-day rolling average
        daily_features = daily_features.withColumn(
            f"{tag}_14d_avg",
            F.avg(mean_col).over(w_14d)
        )
        
        # Degradation slope (7-day): linear regression slope
        # Simplified: (current - 7_days_ago) / 7
        daily_features = daily_features.withColumn(
            f"{tag}_7d_slope",
            (F.col(mean_col) - F.lag(mean_col, 7).over(Window.partitionBy("asset_id").orderBy("calendar_date"))) / 7.0
        )

print(f"   ✓ Added rolling trends for {len(KEY_TAGS)} key tags")
print(f"   ✓ Current feature count: {len(daily_features.columns)} columns")

# 3. Cumulative operating hours (days above threshold)
print("\n3. Computing cumulative operating hours...")

# Define "operating" as days where mean power/load > threshold
# (Adjust tag name based on your schema)
POWER_TAG = tag_list[0]  # Assuming first tag is power-related
OPERATING_THRESHOLD = 50.0  # MW or relevant unit

if f"{POWER_TAG}_daily_mean" in daily_features.columns:
    daily_features = daily_features.withColumn(
        "is_operating",
        F.when(F.col(f"{POWER_TAG}_daily_mean") > OPERATING_THRESHOLD, 1).otherwise(0)
    )
    
    # Cumulative sum of operating days
    daily_features = daily_features.withColumn(
        "cumulative_operating_days",
        F.sum("is_operating").over(
            Window.partitionBy("asset_id").orderBy("calendar_date").rowsBetween(Window.unboundedPreceding, 0)
        )
    )

# 4. Temporal features
print("\n4. Adding temporal features...")

daily_features = daily_features.withColumn(
    "day_of_week",
    F.dayofweek("calendar_date")  # 1=Sunday, 7=Saturday
).withColumn(
    "day_of_month",
    F.dayofmonth("calendar_date")
).withColumn(
    "month",
    F.month("calendar_date")
)

# Days since last stop (join with labels)
labels_for_days_since = spark.table("gold.daily_survival_labels").select(
    "asset_id",
    "calendar_date",
    F.when(F.col("days_to_next_stop") == 0, 1).otherwise(0).alias("is_stop_day")
)

# Self-join to find last stop date
w_lookback = Window.partitionBy("asset_id").orderBy("calendar_date").rowsBetween(Window.unboundedPreceding, -1)

labels_with_last_stop = labels_for_days_since.withColumn(
    "last_stop_date",
    F.when(F.col("is_stop_day") == 1, F.col("calendar_date")).otherwise(F.lit(None))
).withColumn(
    "last_stop_date",
    F.last("last_stop_date", ignorenulls=True).over(w_lookback)
).withColumn(
    "days_since_last_stop",
    F.when(F.col("last_stop_date").isNotNull(), 
           F.datediff(F.col("calendar_date"), F.col("last_stop_date"))
    ).otherwise(999)  # No prior stop
)

# Join back to features
daily_features = daily_features.join(
    labels_with_last_stop.select("asset_id", "calendar_date", "days_since_last_stop"),
    ["asset_id", "calendar_date"],
    "left"
)

print(f"   ✓ Added temporal features")
print(f"   ✓ Final feature count: {len(daily_features.columns)} columns")

# 5. Join with labels to create training dataset
print("\n5. Joining with labels...")

labels = spark.table("gold.daily_survival_labels")

training_data_daily = labels.join(
    daily_features,
    ["asset_id", "calendar_date"],
    "inner"
)

print(f"   ✓ Joined labels: {training_data_daily.count():,} rows")

# Drop rows with excessive nulls (early days without full history)
initial_count = training_data_daily.count()
training_data_daily = training_data_daily.dropna(subset=['days_to_next_stop'])  # Only drop rows missing the target label
final_count = training_data_daily.count()

print(f"   ✓ Dropped rows with >30% nulls: {initial_count - final_count:,} rows removed")
print(f"   ✓ Final training dataset: {final_count:,} rows × {len(training_data_daily.columns)} columns")

# Save to Delta table
training_data_daily.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("ml.training_longterm")

print(f"\n✓ Saved ml.training_longterm")
print(f"\nSchema preview:")
training_data_daily.printSchema()

print(f"\nFeature summary (sample asset):")
sample_asset = TARGET_ASSETS[0]
training_data_daily.filter(F.col("asset_id") == sample_asset) \
    .orderBy(F.desc("calendar_date")) \
    .select("calendar_date", "days_to_next_stop", "label_stop_7d", "days_since_last_stop") \
    .show(20, truncate=False)

## Phase 3b: Train Regression Model

Train a regression model to predict `days_to_next_stop` (0-14 days).

**Model approach:**
- **Regression target**: `days_to_next_stop` (continuous)
- **Algorithms**: RandomForest Regressor, GradientBoosting Regressor
- **Evaluation**: MAE (mean absolute error), RMSE, R²
- **Validation**: 80/20 temporal split (last 20% of dates as holdout)
- **MLflow**: Track experiments for comparison with short-term models

In [ ]:
# Phase 3b: Train Long-Term Survival Regression Model
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Set MLflow experiment
mlflow.set_experiment("Phase3-LongTerm-Survival")

print("="*70)
print(f"PHASE 3b: LONG-TERM SURVIVAL MODEL TRAINING ({HORIZON_DAYS}-DAY PREDICTION)")
print("="*70)

# Load training data
training_df = spark.table("ml.training_longterm").toPandas()

print(f"\n✓ Loaded training data: {len(training_df):,} rows × {len(training_df.columns)} columns")
print(f"  Date range: {training_df['calendar_date'].min()} to {training_df['calendar_date'].max()}")

# Temporal train/test split (80/20 by date)
training_df = training_df.sort_values("calendar_date")
split_idx = int(len(training_df) * 0.8)
train_df = training_df.iloc[:split_idx]
test_df = training_df.iloc[split_idx:]

print(f"\n✓ Train/Test split (temporal):")
print(f"  Train: {len(train_df):,} rows ({train_df['calendar_date'].min()} to {train_df['calendar_date'].max()})")
print(f"  Test:  {len(test_df):,} rows ({test_df['calendar_date'].min()} to {test_df['calendar_date'].max()})")

# Separate features and target
exclude_cols = ['asset_id', 'calendar_date', 'days_to_next_stop', 'censored', 
                'label_stop_7d', 'label_stop_14d', 'days_since_last_stop',
                'cumulative_operating_days']
feature_cols = [c for c in training_df.columns if c not in exclude_cols]

X_train = train_df[feature_cols]
y_train = train_df['days_to_next_stop']

X_test = test_df[feature_cols]
y_test = test_df['days_to_next_stop']

# Use training set median for imputation (not 0, which can be a real sensor value)
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)  # Use TRAIN medians for test too

print(f"\n✓ Feature matrix: {len(feature_cols)} features")
print(f"  Top features: {feature_cols[:10]}")

# Feature selection: top 25 by mutual information (prevent overfitting with 244 features)
# Fast pre-filter: drop zero-variance cols, keep top 100 by F-test, then MI on those
from sklearn.feature_selection import f_regression, mutual_info_regression

nonzero_cols = [c for c in feature_cols if X_train[c].std() > 0]
print(f"Pre-filter: {len(feature_cols)} features -> {len(nonzero_cols)} non-zero variance")

f_scores, _ = f_regression(X_train[nonzero_cols].fillna(0), y_train)
f_ranking = pd.Series(f_scores, index=nonzero_cols).sort_values(ascending=False)
top_candidates = f_ranking.head(100).index.tolist()
print(f"F-test filter: top 100 candidates selected (fast linear pre-screen)")

# Now run MI only on the 100 candidates (10x faster than 1,266)

print(f"Feature selection: {len(feature_cols)} candidates \u2192 selecting top 25")
mi_scores = mutual_info_regression(X_train[top_candidates].fillna(0), y_train, random_state=42)
mi_ranking = pd.Series(mi_scores, index=top_candidates).sort_values(ascending=False)

TOP_N_FEATURES = 25
selected_features = mi_ranking.head(TOP_N_FEATURES).index.tolist()
print(f"\u2713 Selected {len(selected_features)} features by mutual information:")
for f in selected_features[:10]:
    print(f"  {f}: MI = {mi_ranking[f]:.4f}")

X_train = X_train[selected_features]
X_test = X_test[selected_features]
feature_cols = selected_features
print(f"\nTraining with {len(feature_cols)} features ({X_train.shape[0]} rows)")

print(f"\n✓ Target distribution (days_to_next_stop):")
print(f"  Train: mean={y_train.mean():.2f}, median={y_train.median():.2f}, std={y_train.std():.2f}")
print(f"  Test:  mean={y_test.mean():.2f}, median={y_test.median():.2f}, std={y_test.std():.2f}")

from sklearn.model_selection import TimeSeriesSplit, cross_val_score

# Train multiple regression models
models_to_train = {
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        max_depth=5,
        min_samples_split=30,
        min_samples_leaf=20,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        min_samples_split=30,
        min_samples_leaf=20,
        subsample=0.8,
        random_state=42
    )
}

results = []

tscv = TimeSeriesSplit(n_splits=5)
for name, model in models_to_train.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=tscv, scoring='neg_mean_absolute_error')
    print(f"  {name} 5-fold CV MAE: {-cv_scores.mean():.3f} \u00b1 {cv_scores.std():.3f}")

for model_name, model in models_to_train.items():
    print(f"\n{'='*70}")
    print(f"Training {model_name}...")
    print(f"{'='*70}")
    
    with mlflow.start_run(run_name=f"LongTerm-{model_name}"):
        # Log parameters
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("horizon", f"{HORIZON_DAYS}_days")
        mlflow.log_param("n_features", len(feature_cols))
        mlflow.log_param("train_samples", len(X_train))
        mlflow.log_param("test_samples", len(X_test))
        
        # Train model
        model.fit(X_train, y_train)
        
        # Predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Clip predictions to [0, 14] range
        y_train_pred = np.clip(y_train_pred, 0, HORIZON_DAYS)
        y_test_pred = np.clip(y_test_pred, 0, HORIZON_DAYS)
        
        # Calculate metrics
        train_mae = mean_absolute_error(y_train, y_train_pred)
        test_mae = mean_absolute_error(y_test, y_test_pred)
        
        train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
        test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
        
        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)

        # Uncensored-only evaluation (more meaningful — excludes ceiling-capped values)
        if 'censored' in test_df.columns:
            uncensored_mask = test_df['censored'] == 0
            if uncensored_mask.sum() > 0:
                y_test_unc = y_test[uncensored_mask.values]
                y_pred_unc = y_test_pred[uncensored_mask.values]
                unc_mae = mean_absolute_error(y_test_unc, y_pred_unc)
                unc_r2 = r2_score(y_test_unc, y_pred_unc)
                print(f"  Uncensored-only ({uncensored_mask.sum()} rows): MAE={unc_mae:.3f}, R\u00b2={unc_r2:.3f}")
        
        # Log metrics
        mlflow.log_metric("train_mae", train_mae)
        mlflow.log_metric("test_mae", test_mae)
        mlflow.log_metric("train_rmse", train_rmse)
        mlflow.log_metric("test_rmse", test_rmse)
        mlflow.log_metric("train_r2", train_r2)
        mlflow.log_metric("test_r2", test_r2)
        
        # Log model
        mlflow.sklearn.log_model(model, "model")
        
        run_id = mlflow.active_run().info.run_id
        
        print(f"\n✓ Model trained and logged to MLflow")
        print(f"  Run ID: {run_id}")
        print(f"\n  Train MAE: {train_mae:.3f} days | RMSE: {train_rmse:.3f} | R²: {train_r2:.3f}")
        print(f"  Test  MAE: {test_mae:.3f} days | RMSE: {test_rmse:.3f} | R²: {test_r2:.3f}")
        
        # Feature importance (for tree-based models)
        if hasattr(model, 'feature_importances_'):
            feature_importance = pd.DataFrame({
                'feature': feature_cols,
                'importance': model.feature_importances_
            }).sort_values('importance', ascending=False).head(15)
            
            print(f"\n  Top 15 Features:")
            for idx, row in feature_importance.iterrows():
                print(f"    {row['feature']}: {row['importance']:.4f}")
        
        results.append({
            'model': model_name,
            'run_id': run_id,
            'test_mae': test_mae,
            'test_rmse': test_rmse,
            'test_r2': test_r2
        })

# Summary
print(f"\n{'='*70}")
print("TRAINING COMPLETE - MODEL COMPARISON")
print(f"{'='*70}")

results_df = pd.DataFrame(results).sort_values('test_mae')
print(f"\n{results_df.to_string(index=False)}")

best_model = results_df.iloc[0]
print(f"\n🏆 Best model: {best_model['model']}")
print(f"   Run ID: {best_model['run_id']}")
print(f"   Test MAE: {best_model['test_mae']:.3f} days")
print(f"   Test RMSE: {best_model['test_rmse']:.3f} days")
print(f"   Test R²: {best_model['test_r2']:.3f}")

print(f"\n💡 Interpretation:")
print(f"   - MAE {best_model['test_mae']:.1f} days means predictions are off by ~{best_model['test_mae']:.1f} days on average")
print(f"   - R² {best_model['test_r2']:.3f} means the model explains {best_model['test_r2']*100:.1f}% of variance in stop timing")
print(f"   - Use this model to predict maintenance windows {HORIZON_DAYS} days out")

## Phase 4b: Daily Scoring Pipeline

Score the latest day's sensor data to predict probability of stop over next 14 days.

**Output:**
- **Table**: `ml.predictions_longterm`
- **Columns**: 
  - `scoring_timestamp` - when prediction was made
  - `asset_id` - equipment identifier
  - `predicted_days_to_stop` - model's prediction (0-14)
  - `confidence_interval_low/high` - uncertainty bounds (if using ensemble)
  - `risk_level` - categorical (CRITICAL <3d, HIGH 3-7d, MEDIUM 7-14d, LOW >14d)
  - `model_run_id` - MLflow run for traceability

In [ ]:
# Phase 4b: Daily Scoring with Long-Term Survival Model
from datetime import datetime
import mlflow.sklearn

print("="*70)
print(f"PHASE 4b: LONG-TERM SURVIVAL SCORING ({HORIZON_DAYS}-DAY PREDICTION)")
print("="*70)

# CONFIGURATION: Set the model run ID from Phase 3b training
# TODO: Replace with your best model's run_id after training
MODEL_RUN_ID = "REPLACE_WITH_BEST_RUN_ID"  # Example: "abc123-def456-..."

# Alternative: Auto-detect best model from MLflow
if MODEL_RUN_ID == "REPLACE_WITH_BEST_RUN_ID":
    print("\n⚠️  MODEL_RUN_ID not set. Searching for best model in MLflow...")
    
    client = mlflow.tracking.MlflowClient()
    experiment = client.get_experiment_by_name("Phase3-LongTerm-Survival")
    
    if experiment:
        runs = client.search_runs(
            experiment_ids=[experiment.experiment_id],
            filter_string="params.model_type != '' AND metrics.test_mae > 0",
            order_by=["metrics.test_mae ASC"],
            max_results=1
        )
        
        if runs:
            best_run = runs[0]
            MODEL_RUN_ID = best_run.info.run_id
            test_mae_raw = best_run.data.metrics.get('test_mae', None)
            test_mae = float(test_mae_raw) if test_mae_raw is not None else None
            model_type = best_run.data.params.get('model_type', 'Unknown')
            
            print(f"   ✓ Auto-selected best model:")
            print(f"     Run ID: {MODEL_RUN_ID}")
            print(f"     Model: {model_type}")
            print(f"     Test MAE: {test_mae:.3f} days" if test_mae else "     Test MAE: N/A")
        else:
            raise ValueError("No trained models found in MLflow experiment 'Phase3-LongTerm-Survival'")
    else:
        raise ValueError("MLflow experiment 'Phase3-LongTerm-Survival' not found. Run Phase 3b first.")

# Load model
print(f"\n✓ Loading model from MLflow...")
model_uri = f"runs:/{MODEL_RUN_ID}/model"
model = mlflow.sklearn.load_model(model_uri)
print(f"  Model loaded: {type(model).__name__}")

# Safety check: ensure this is a regression model, not an autolog artifact
if not hasattr(model, 'predict'):
    raise ValueError(f"Loaded model is {type(model).__name__} which has no predict(). This is likely an MLflow autolog artifact (e.g. KNNImputer). Re-run Phase 3b training to create a proper model run.")

# Load today's features (most recent date in training_dataset_daily_survival)
training_data = spark.table("ml.training_longterm")
latest_date = training_data.agg(F.max("calendar_date")).collect()[0][0]

print(f"\n✓ Latest available data: {latest_date}")

# Get features for latest date
scoring_data = training_data.filter(F.col("calendar_date") == latest_date).toPandas()

print(f"  Assets to score: {len(scoring_data)}")
print(f"  Assets: {scoring_data['asset_id'].tolist()}")

# Prepare features (same as Phase 3b)
exclude_cols = ['asset_id', 'calendar_date', 'days_to_next_stop', 'censored',
                'label_stop_7d', 'label_stop_14d', 'days_since_last_stop',
                'cumulative_operating_days']
feature_cols = [c for c in scoring_data.columns if c not in exclude_cols]

X_score = scoring_data[feature_cols].fillna(0)

# Align features to what the model was trained with
if hasattr(model, 'feature_names_in_'):
    trained_features = list(model.feature_names_in_)
    # Add any missing columns (with 0), drop any extra columns
    for f in trained_features:
        if f not in X_score.columns:
            X_score[f] = 0
            print(f"  ⚠️ Added missing feature '{f}' (filled with 0)")
    X_score = X_score[trained_features]
    print(f"  Aligned to {len(trained_features)} model features")

print(f"\n✓ Feature matrix: {len(feature_cols)} features")

# Generate predictions
predictions = model.predict(X_score)
predictions = np.clip(predictions, 0, HORIZON_DAYS)

print(f"\n✓ Predictions generated:")
for i, asset in enumerate(scoring_data['asset_id']):
    pred = predictions[i]
    
    # Risk categorization
    if pred <= 3:
        risk = "CRITICAL"
    elif pred <= 7:
        risk = "HIGH"
    elif pred <= 14:
        risk = "MEDIUM"
    else:
        risk = "LOW"
    
    print(f"  {asset}: {pred:.1f} days ({risk})")

# Build output dataframe
scoring_results = pd.DataFrame({
    'scoring_timestamp': datetime.now(),
    'scoring_date': latest_date,
    'asset_id': scoring_data['asset_id'],
    'predicted_days_to_stop': predictions,
    'risk_level': [
        "CRITICAL" if p <= 3 else "HIGH" if p <= 7 else "MEDIUM" if p <= 14 else "LOW"
        for p in predictions
    ],
    'model_type': 'longterm_survival',
    'horizon': f'{HORIZON_DAYS}_days',
    'model_run_id': MODEL_RUN_ID
})

# For tree-based models, we can estimate confidence using tree variance
if hasattr(model, 'estimators_'):
    # Get predictions from each tree (GBR stores trees as 2D array [n_estimators, 1])
    estimators = model.estimators_.ravel() if hasattr(model.estimators_, 'ravel') else model.estimators_
    all_tree_preds = np.array([tree.predict(X_score) for tree in estimators])
    
    # Calculate std dev across trees as uncertainty estimate
    pred_std = np.std(all_tree_preds, axis=0)
    
    scoring_results['prediction_std'] = pred_std
    scoring_results['confidence_interval_low'] = np.clip(predictions - 1.96 * pred_std, 0, HORIZON_DAYS)
    scoring_results['confidence_interval_high'] = np.clip(predictions + 1.96 * pred_std, 0, HORIZON_DAYS)
    
    print(f"\n✓ Confidence intervals calculated (95% CI):")
    for i, asset in enumerate(scoring_data['asset_id']):
        print(f"  {asset}: [{scoring_results.iloc[i]['confidence_interval_low']:.1f}, "
              f"{scoring_results.iloc[i]['confidence_interval_high']:.1f}] days")

# Convert to Spark and save
scoring_results_spark = spark.createDataFrame(scoring_results)

# Append to long-term predictions table
scoring_results_spark.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable("ml.predictions_longterm")

print(f"\n✓ Saved predictions to ml.predictions_longterm")
print(f"\n{'='*70}")
print(f"SCORING COMPLETE")
print(f"{'='*70}")

# Display summary
print(f"\nPrediction Summary ({latest_date}):")
print(scoring_results[['asset_id', 'predicted_days_to_stop', 'risk_level']].to_string(index=False))

print(f"\n💡 Next Steps:")
print(f"   1. Schedule this notebook to run daily (e.g., 6 AM)")
print(f"   2. Create alerts for CRITICAL/HIGH risk assets")
print(f"   3. Compare with short-term model (Phase 4) for triangulation")
print(f"   4. Feed predictions into maintenance scheduling system")

## Integration with Short-Term Model

### Dual-Horizon Prediction Strategy

| Scenario | Short-Term (4-24h) | Long-Term (14-day) | Action |
|----------|-------------------|-------------------|--------|
| **Both LOW** | No stop predicted in 24h | No stop in 14d | ✅ Normal operations |
| **Long-term HIGH, Short-term LOW** | No imminent stop | Stop likely in 7-14d | 📋 Schedule maintenance, order parts |
| **Short-term CRITICAL** | Stop in <4h | (any) | 🚨 Emergency response, shutdown prep |
| **Both HIGH** | Stop in 8-24h | Stop in 3-7d | ⚠️ Controlled shutdown + maintenance |

### Calibration Check

Run both models on the same test period and compare:

```python
# Compare predictions for May 2026
short_term = spark.table("ml.predictions_shortterm").filter(
    (F.col("scoring_timestamp") >= "2026-05-01") & 
    (F.col("horizon") == "4h")
)

long_term = spark.table("ml.predictions_longterm").filter(
    F.col("scoring_date") >= "2026-05-01"
)

# Join on asset + date
# Check: Does long_term predict stops that short_term catches 4h before?
```

### Operational Deployment

**Daily workflow:**
1. **6:00 AM** - Run Phase-LongTerm-Survival → get 14-day outlook
2. **Every 15 min** - Run Phase4-Model-Scoring-Batch → get 4h/8h/24h alerts
3. **When long-term shows HIGH risk:**
   - Notify maintenance planner
   - Check parts inventory
   - Review crew schedule
4. **When short-term shows CRITICAL:**
   - Alert operations immediately
   - Prepare backup equipment
   - Initiate shutdown protocol

### Model Monitoring

Track these metrics weekly:
- **MAE drift**: Is prediction error increasing over time?
- **Calibration**: Do "7-day" predictions actually stop in ~7 days?
- **False alarm rate**: How often do CRITICAL predictions not result in stops?
- **Coverage**: Did the model predict all actual stops (recall)?